In [9]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [13]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [14]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [18]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Példányosítsd a modellt explicit módon
# Használj létező nevet: "gemini-1.5-flash" vagy "gemini-2.0-flash-exp"
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt
)

In [19]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [20]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='87ab2e36-d032-4050-a16b-502a74086e80'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'8b6bb65e-03e1-48bf-8e69-21983a2641b8': 'CrIBAb4+9vuotPL+wHxAJzZ4Se9y3Y0DyEKDYk5jA0MCqd+phQ86QNbHuYzMvEr+YrA6Mt/cx1Ac0AHahSXJHEmHTfciw4I5VfZOQa9r5+CYblO4seYjCbD3JkqgkPHdk79SSqopD53dqp2Fh/u8ZhIlRC4FRvaXwuuSp/GlWqxNxIP2pU0mEWzWx2cuSAgqDyQy+by0ut+MBSu8ZRWndUbjSm9wgmQj8atD7MDFT/Fm9iH7fw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d7280-3b89-7452-bf47-1a0710dd97d0-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '8b6bb65e-03e1-48bf-8e69-21983a2641b8', 'type': '

## Online MCP

In [21]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [23]:
agent = create_agent(
    model=llm,
    tools=tools,
)

In [24]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='1de470cf-007b-44cf-818f-c9a7e2645d7b'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "America/New_York"}'}, '__gemini_function_call_thought_signatures__': {'435804a2-1f26-4164-8fe7-92629a9899e7': 'CuwCAb4+9vsqhAIeT6ohlTi3hNiFlSa77/B1yz+6aY9cLZHw8PT+KUgCDgI0316qxCMvGwXcTroD2reKX29f5Mi8HUne397tewOHa9F3+UHIW0ZiHFr8CPlwV0kfYbyeKBgGchNpTm4/kFBDUwCITGrkMXF+Alx/RYDkloKBY1BNV+l0bLtg2hs2RjwdqdGFGa4KFJTahCs9o8yiq0WUOsC7uNVh2ZTLHcUeT44GLEj/Lcr0oWzXBWG3opTWxWDfydvzHz1J17fTsNlAJztuT6CV1VhIfQigrUK1QjaDW7WdfxGOD4KQ9QfVaHMAfwiJdUs46KdDhudNs4/ORgc47W0Yw3+qjbDeSSpvtZuofoHwx1XGT9/kYl5VIbcmj+0zKpfUoJri1gP8So0HctfRQiPdv0+OVhrHijCCgurN+ECqh3DDOS5kMpBV0VQB0ABqCY3XLR34HKlnHFfs12ryCNqAyIjJ1crok2m4hQEdnw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'g